In [1]:
# from modules import *
from pathlib import Path
import cmocean
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from obspy.geodetics.base import gps2dist_azimuth,degrees2kilometers,locations2degrees
import pygmt
import pandas as pd
from scipy.interpolate import griddata
import matplotlib.colors as mcolors
from scipy.spatial import KDTree
from shapely.geometry import LineString, Point
from scipy.signal import detrend
import pygeodesy
from pygeodesy.ellipsoidalKarney import LatLon
import itertools
from collections.abc import Iterable
ginterpolator = pygeodesy.GeoidKarney("/Users/charlesh/Documents/Codes/OBS_Methods/NOISE/Research/Notebooks/egm2008-5.pgm")
geoid= lambda lat,lon: ginterpolator(LatLon(lat, lon))
datafolder = Path('/Users/charlesh/Documents/Codes/OBS_Methods/NOISE/Research/Notebooks/632.Galapagos.GravData')
plotfold = datafolder/'ProjFigs'
gravdatafile = datafolder/'Processed_Daily_Grav'/'grv_all_sat.txt'
line_points = np.array([[-98, 2.249885], [-94,2.5],[-90.907174, 1.849262], [-90.707551, 0.921334], [-85.379041, 0.633260]])
def unravel(nested_list):
    flat_list = []
    for item in nested_list:
        test=(isinstance(item, Iterable)) and (not isinstance(item, str))
        if test:flat_list.extend(unravel(item))
        else:flat_list.append(item)
    return flat_list
def ridge_distance(lola):
    line_points = np.array([[-98, 2.249885], [-94,2.5],[-90.907174, 1.849262], [-90.707551, 0.921334], [-85.379041, 0.633260]])
    line = LineString(line_points)
    point = Point(lola)
    closest_point_on_line = line.interpolate(line.project(point))
    shortest_distance, _, _ = gps2dist_azimuth(lola[1], lola[0], closest_point_on_line.y, closest_point_on_line.x)
    p=-1 if point.y>closest_point_on_line.y else 1
    # print(f"Closest point on the line: {closest_point_on_line}")
    # print(f"Shortest distance to the line: {shortest_distance:.2f} meters")
    return p*shortest_distance
wgs84 = lambda lat,a=6378.137,f=1/298.257222:a*((1+(((2*f)-(f**2))/((1-f)**2)) * (np.sin(np.deg2rad(lat)))**2  )**(-0.5))
def df_map(key='FA_Anomaly_mGal',clim=None,size=15):
    grid = grid_data.pivot(index='y', columns='x', values='z')
    x,y,z= grid_data.x,grid_data.y,grid_data.z
    # Define a diverging normalization centered at zero
    norm = mcolors.TwoSlopeNorm(vmin=.6*grid.values.min(), vcenter=0, vmax=.3*grid.values.max())
    custom_cmap =  cmocean.cm.balance
    # Plot the grid using imshow
    plt.figure(figsize=(15, 15))
    plt.imshow(grid.values,extent=[x.min(), x.max(), y.min(), y.max()],origin='lower',cmap=custom_cmap,norm=norm)
    # plt.colorbar(label='Elevation (m)', shrink=0.75, aspect=30)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    top=r'Moho ($\Delta\rho=600 kg/m^{3}$) depth correction using the ' 
    if key=='MohoDepth_Using_Geoid_meters':title=top+'Geoid'
    elif key=='MohoPerturb_Using_GD_meters':title=top+'Disturbance anomaly'
    elif key=='MohoPerturb_Using_FA_meters':title=top+'Free-Air anomaly'
    elif key=='GD_Anomaly_mGal':title='Gravity Disturbance (GD) Anomaly (mGal)'
    elif key=='FA_Anomaly_mGal':title='Free-Air (FA) Anomaly (mGal)'
    elif key=='Geoid_Anomaly_meters':title='Geoid Anomaly (HG), height (m) above the ellipsoid'
    else:title=key
    plt.title(title)
    plt.xlim([x.min(), x.max()])
    plt.ylim([y.min(), y.max()])
    plt.grid('major',alpha=.2,c='k')
    plt.plot(line_points[:, 0], line_points[:, 1], linewidth=2, color='w', zorder=10,linestyle=':',alpha=0.4)  # Adjust zorder to ensure it's on top
    xx=df.Lon;yy=df.Lat
    fillcolor = np.array([1, 0, 1, 1])  # RGBA for fillcolor
    zz=df[key]
    vcenter=0 if ((df[key].min()<0)&(df[key].max()>0)) else zz.mean()
    if key=='FA_Anomaly_mGal':cmap='jet';clim=[-20,20];clim_extended=[-60,80];colorres=20;cmap=plt.colormaps[cmap].resampled(colorres);norm = mcolors.Normalize(vmin=-25, vmax=25)  # Set color limits
    elif key.lower()=='bathymetry':cmap='rainbow';clim=[zz.min(),zz.max()];clim_extended=clim;colorres=256;cmap=plt.colormaps[cmap].resampled(colorres).reversed();norm=None
    elif key=='Geoid_Anomaly_meters':cmap='rainbow';clim=[-10,10];clim_extended=clim;colorres=256;cmap=plt.colormaps[cmap].resampled(colorres);norm=mcolors.TwoSlopeNorm(vmin=clim_extended[0], vcenter=0, vmax=clim_extended[1])
    elif key=='MohoDepth_Using_Geoid':cmap='rainbow';clim=[-200,200];clim_extended=clim;colorres=256;cmap=plt.colormaps[cmap].resampled(colorres);norm=mcolors.TwoSlopeNorm(vmin=clim_extended[0], vcenter=0, vmax=clim_extended[1])
    elif key=='MohoPerturb_Using_GD_meters':cmap='rainbow';clim=[-400,400];clim_extended=clim;colorres=256;cmap=plt.colormaps[cmap].resampled(colorres);norm=mcolors.TwoSlopeNorm(vmin=clim_extended[0], vcenter=0, vmax=clim_extended[1])
    elif key=='MohoPerturb_Using_FA_meters':cmap='rainbow';clim=[-400,400];clim_extended=clim;colorres=256;cmap=plt.colormaps[cmap].resampled(colorres);norm=mcolors.TwoSlopeNorm(vmin=clim_extended[0], vcenter=0, vmax=clim_extended[1])
    elif key=='GD_Anomaly_mGal':cmap='jet';clim=[-20,20];clim_extended=[-60,80];colorres=20;cmap=plt.colormaps[cmap].resampled(colorres);norm = mcolors.Normalize(vmin=-25, vmax=25)
    else:
        if clim==None:cmap='rainbow';clim=[None,None];clim_extended=clim;colorres=256;cmap=plt.colormaps[cmap].resampled(colorres);norm=None
    # cmap=plt.colormaps[cmap].resampled(colorres).reversed()
    # colors = cmap(np.linspace(0, 1, colorres))
    # idx = np.logical_and(np.linspace(clim_extended[0], clim_extended[1], colorres) >= -crossval,np.linspace(clim_extended[0], clim_extended[1], colorres) <= crossval)
    # colors[idx] = fillcolor
    # norm = mcolors.TwoSlopeNorm(vmin=clim[0], vcenter=vcenter, vmax=clim[1])
    # cmap = ListedColormap(colors)
    sc=plt.scatter(xx,yy,s=size,c=zz,cmap=cmap,zorder=11,norm=norm)
    plt.gca().set_aspect('equal')
    if key.lower().find('mgal')>=0:units='mGal'
    elif key.lower().find('depth')>=0:units='Meters'
    elif key.lower().find('meters')>=0:units='Meters'
    else:units=''
    # if key=='FA_Anomaly_mGal':
    #     cbar = plt.colorbar(plt.scatter([],[],s=0,c=[],cmap=cmap,
    #     norm=mcolors.TwoSlopeNorm(vmin=clim_extended[0], vcenter=vcenter, vmax=clim_extended[1]),zorder=0),
    #     label=units, shrink=0.4, aspect=20, extend='both',pad=0.01)
    #     # cbar.set_ticks([-60, -40, -20, 0, 20, 40, 60, 80])
    # elif key=='MohoDepth_Using_Geoid':
    #     cbar = plt.colorbar(sc,label=units, shrink=0.5, aspect=30, extend='both',pad=0.01)
    # else: 
    cbar = plt.colorbar(sc,label=units, shrink=0.5, aspect=30, extend='both',
    norm=mcolors.TwoSlopeNorm(vmin=clim_extended[0], vcenter=vcenter, vmax=clim_extended[1]),pad=0.01)
def df_scatter(keys,colorkey='Geoid_Anomaly_meters',cmap=None):
    fontprops={'fontweight':'bold','fontsize':9}
    colorlabel = colorkey.replace('FA_Anomaly_mGal','Free Air Anomaly').replace('Bathymetry','Batymetry,m').replace('Geoid_Anomaly_meters','Geoid Height,m')
    nrows,ncols=int(np.ceil(len(keys)/2)),int(np.floor(len(keys)/2))
    figsize = (7.5*ncols,3.5*nrows)
    fig,axes = plt.subplots(nrows,ncols,figsize=figsize)
    axes=axes.reshape(-1)
    labels={'Lon':'Longitude, °','Lat':'Latitude, °','mgal':'Free-air anomaly, mGal','FA_Anomaly_mGal':'Free-air anomaly, mGal',
    'Path':'Survey Path distance, km','Geoid_Anomaly_meters':'Geoid Height, m','Bathymetry':'Bathymetry, m','EllipsoidHeight':'Ellipsoid Height, km','MohoDepth_Using_Geoid_meters':'Crustal Thickness Perturbation, meters',
    'MohoDepth_Using_Geoid_meters':'Crustal Thickness Perturbation, meters','MohoPerturb_Using_FA_meters':'Crustal Thickness Perturbation, meters','MohoPerturb_Using_GD_meters':'Crustal Thickness Perturbation, meters'}
    colors=df[colorkey].values
    if cmap:cmap='rainbow' if ((len(colors[colors<0])>0)&(len(colors[colors>0])>0)) else 'turbo'
    for ax,(x,y) in zip(axes,keys):
        sc3=ax.scatter(df[x],df[y],s=1,c=df[colorkey].values,cmap=cmap)
        ax.axhline(0,linestyle=':',c='k')
        ax.grid(alpha=0.4)
        ax.set_xlabel(labels[x],**fontprops)
        ax.set_ylabel(labels[y],labelpad=-1,**fontprops)
    cbar_ax = fig.add_axes([0.92, 0.10, 0.01, 0.7])
    cbar=fig.colorbar(sc3, cax=cbar_ax, label=colorlabel,aspect=90,shrink=.3)
    cbar.ax.set_ylabel(labels[colorkey], fontweight='bold', fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    for label in cbar.ax.get_yticklabels():label.set_fontweight('bold')

In [ ]:
# labels = ['Year','CumulativeEQs']
# file = Path('/Users/charlesh/Downloads/black_curve_full.csv')
# data=pd.read_csv(file,header=0,names=labels)
# black = data.iloc[::31].reset_index(drop=True)
# black = black.iloc[:2000]
# file = Path('/Users/charlesh/Downloads/green_curve_full.csv')
# data=pd.read_csv(file,header=0,names=labels)
# green = data.iloc[::11].reset_index(drop=True)
# green = green.iloc[:2000].reset_index(drop=True)
# file = Path('/Users/charlesh/Downloads/blue_curve_full.csv')
# data=pd.read_csv(file,header=0,names=labels)
# blue = data.iloc[::10].reset_index(drop=True)
# blue = blue.iloc[:2000].reset_index(drop=True)

# # pd.concat([black.iloc[:2000],green.iloc[:2000],blue.iloc[:2000]])
# # black.iloc[:2000]
# black
# # green
# # blue
# x=black.Year.values
# y=black.CumulativeEQs.values
# # x=green.Year.values
# # y=green.CumulativeEQs.values
# # x=blue.Year.values
# # y=blue.CumulativeEQs.values
# plt.scatter(x,y)

In [ ]:
# {'Namie':[171.0,44.0],'Tuhoku':[140.0,29.0],'Ishinomaki':[206,43],'Onagawa':[200,43.0]}

In [3]:
import xarray as xr
from pathlib import Path
fold = Path('/Users/charlesh/Downloads/Plate2_Japan/')
# Step 1: Load the slab2_dep.grd
ds = xr.open_dataset(str(fold/'kur_slab2_dep_02.24.18.grd'))

In [ ]:
# cmcrameri.show_cmaps()

In [ ]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator
import matplotlib.pyplot as plt

import xarray as xr
from pathlib import Path
import cmcrameri
from pyproj import Geod
fold = Path('/Users/charlesh/Downloads/Plate2_Japan/')
lines = dict()
# lines['Namie']=[[143.428734,36.752632],[140.938875,38.074183]]
lines['Tuhoku']=[[143.965295,38.152632],[141.021313,38.361032]]
lines['Namie']=[[143.428734,36.752632],[140.938875,38.074183]]
lines['Ishinomaki']=[[143.965295,38.152632],[141.022554,38.507958]]
lines['Onagawa']=[[143.704341, 37.466506],[140.987658,38.359843]]
# Plot using real distance
# plt.figure(figsize=1.3*np.array([10, 6]))
fig = plt.figure(figsize=np.array([5, 5]))
# Step 1: Load the slab2_dep.grd
ds = xr.open_dataset(str(fold/'kur_slab2_dep_02.24.18.grd'))
ds_thick = xr.open_dataset(str(fold/'kur_slab2_thk_02.24.18.grd'))
# Inspect variable name if needed
# print(ds)
# Assume the data variable is named 'z' or similar
depth = ds['z']
thick = ds['z']
# Get the grid coordinates
lons = depth['x'].values
lats = depth['y'].values
depth_values = depth.values
thick_values = thick.values
# Set up an interpolator
method = 'linear'
# method = None
interp_func = RegularGridInterpolator((lats, lons), depth_values, bounds_error=False, fill_value=np.nan,method=method)
interp_func_thick = RegularGridInterpolator((thick['y'].values, thick['x'].values), thick_values, bounds_error=False, fill_value=np.nan,method=method)

colors = cmcrameri.cm.managua.resampled(4).reversed()
colors = {k:colors(ki) for ki,k in enumerate(lines.keys())}


colors['Namie'] = [212/255,66/255,70/255]



labels = {'Namie':'Mw 7.1 | Namie','Tuhoku':'Mw 9.1 | Tuhoku','Ishinomaki':'Mw 7.0 | Ishinomaki','Onagawa':'Mw 6.9 | Onagawa'}
keys = ['Tuhoku','Namie','Ishinomaki','Onagawa']
dist_temp = []
dep_temp = []
thick_temp = []
for ji,j in enumerate(keys):
    lon_start, lat_start = lines[j][0]
    lon_end, lat_end = lines[j][1]
    # Define your line (start and end points)
    # lon_start, lat_start = 143.767397,37.528070
    # lon_end, lat_end = 140.915669,  38.344967
    # Number of points along the line
    n_points = 1000
    lons_line = np.linspace(lon_start, lon_end, n_points)
    lats_line = np.linspace(lat_start, lat_end, n_points)
    # Stack into (lat, lon) pairs
    sample_points = np.vstack((lats_line, lons_line)).T
    # Sample depth values
    depth_along_line = interp_func(sample_points)
    depth_along_line = abs(depth_along_line)
    thick_along_line = interp_func_thick(sample_points)
    # Calculate distance along the line (using pyproj.Geod)
    geod = Geod(ellps="WGS84")
    distances = np.zeros(n_points)
    for i in range(1, n_points):
        _, _, d = geod.inv(lons_line[i-1], lats_line[i-1], lons_line[i], lats_line[i])
        distances[i] = distances[i-1] + d / 1000.0  # Convert meters to km
    zorder = ji*50
    scale = 7
    linewidth=scale*((4-ji)/4)

    #The last kilometer is not given in Plate2. Apply a linear interp using the slope of the last two samples.
    if sum(np.isnan(depth_along_line))>0:
        distances = distances[~np.isnan(depth_along_line)]
        thick_along_line = thick_along_line[~np.isnan(depth_along_line)]
        depth_along_line = depth_along_line[~np.isnan(depth_along_line)]

        x=distances
        y=depth_along_line
        slope = np.mean(np.diff(y)[0]/np.diff(x)[0])
        margindepth = y[0]-(x[0]*slope)
        depth_along_line = np.insert(depth_along_line,[0],margindepth)

        y=thick_along_line
        slope = np.mean(np.diff(y)[0]/np.diff(x)[0])
        marginthick = y[0]-(x[0]*slope)
        thick_along_line = np.insert(thick_along_line,[0],marginthick)
        distances = np.insert(distances,[0],0)
    plt.plot(distances, depth_along_line,c=colors[j],linewidth=linewidth,zorder=zorder)

    dist_temp.append(distances.reshape(-1).tolist())
    dep_temp.append(depth_along_line.reshape(-1).tolist())
    thick_temp.append(thick_along_line.reshape(-1).tolist())
    # plt.fill_between(distances,depth_along_line,depth_along_line-thick_along_line,color='darkgrey',zorder=-1000,edgecolor='k' if ji==3 else None)
    print(f'{j}:{zorder}')
plt.fill_between(distances,depth_along_line,depth_along_line-thick_along_line,color='darkgrey',zorder=-1000,edgecolor='k',linewidth=1,alpha=0.4)

fontsize=10

plt.xlabel('Distance Along Line (km)',fontsize=fontsize)
plt.ylabel('Depth to Slab (km)',fontsize=fontsize)
# plt.title('Subducting Slab Depth Along Line',fontsize=fontsize,fontweight='bold')

evs = {'Tuhoku':[140.0,29.0],'Namie':[171.0,44.0],'Ishinomaki':[206,43],'Onagawa':[200,43.0]}

[plt.scatter(evs[k][0],evs[k][1],s=150,label=labels[k],c=colors[k],edgecolors='k',zorder=1000) for ki,k in enumerate(keys)]

[plt.text(evs[k][0],(evs[k][1]+5) if np.isin(ki,[1,3]) else (evs[k][1]-4),f'{int(evs[k][1])}km',zorder=1000,horizontalalignment='center',verticalalignment='top' if np.isin(ki,[1,3]) else 'bottom',bbox=dict(facecolor='white', edgecolor='white', boxstyle='round,pad=0.01')) for ki,k in enumerate(keys) if not ki==3]
keys = ['Tuhoku','Namie','Ishinomaki','Onagawa']
plt.legend(loc='lower left',fontsize=fontsize)
plt.gca().invert_yaxis()  # Deeper depths downward
plt.grid(alpha=0.4)
plt.xlim(0,250)
plt.ylim(125,0)
plt.gca().set_yticks(np.arange(0,130,10))

x=plt.gca().get_xlim()
y=plt.gca().get_ylim()

plt.text(220,5,'VE 2:1',alpha=0.6)

plt.show()
# If you want lat/lon/depth/distance output
plotfold = Path('/Users/charlesh/Downloads/451_Paper')
fig.savefig(plotfold/'PlateDepths.png',dpi=500)

In [ ]:
# distances=np.atleast_1d(distances).reshape(-1)
# depth_along_line=np.atleast_1d(depth_along_line).reshape(-1)

# interp_func = RegularGridInterpolator(distances, depth_along_line, bounds_error=False, fill_value=np.nan,method=method)
# interp_func_thick = RegularGridInterpolator(distances, thick_along_line, bounds_error=False, fill_value=np.nan,method=method)
dists=np.linspace(min(distances),max(distances),1000)

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

# Ensure 1D arrays
distances = np.atleast_1d(distances).reshape(-1)
depth_along_line = np.atleast_1d(depth_along_line).reshape(-1)
thick_along_line = np.atleast_1d(thick_along_line).reshape(-1)
# Create interpolation functions
interp_func = interp1d(distances, depth_along_line, bounds_error=False, fill_value=np.nan, kind=method)
interp_func_thick = interp1d(distances, thick_along_line, bounds_error=False, fill_value=np.nan, kind=method)
# New sample distances
distances = np.linspace(min(distances), max(distances), 1000)
# Interpolated values
depth_along_line = interp_func(dists)
thick_along_line = interp_func_thick(dists)


In [ ]:
np.array([distances.tolist(),depth_along_line.tolist(),depth_along_line.tolist()])
distances=distances[4:]
depth_along_line=depth_along_line[4:]
thick_along_line=thick_along_line[4:]